<div style="
  background: linear-gradient(145deg, #0f172a, #1e293b);
  border: 4px solid transparent;
  border-radius: 14px;
  padding: 18px 22px;
  margin: 12px 0;
  font-size: 26px;
  font-weight: 600;
  color: #f8fafc;
  box-shadow: 0 6px 14px rgba(0,0,0,0.25);
  background-clip: padding-box;
  position: relative;
">
  <div style="
    position: absolute;
    inset: 0;
    padding: 4px;
    border-radius: 14px;
    background: linear-gradient(90deg, #06b6d4, #3b82f6, #8b5cf6);
    -webkit-mask: 
      linear-gradient(#fff 0 0) content-box, 
      linear-gradient(#fff 0 0);
    -webkit-mask-composite: xor;
    mask-composite: exclude;
    pointer-events: none;
  "></div>
  <b>01. ⚡ Concurrent Web File Downloader</b>
</div>


### 📌 Project Overview
In this project, we implement a highly concurrent image and file downloader that dramatically speeds up batch HTTP transfers using thread pooling.
We'll integrate progress tracking visualizers (`tqdm`) and design a custom exponential backoff decorator to deal with unstable network connections.

#### 🔑 Key Concepts Covered:
- Utilizing Python's `concurrent.futures.ThreadPoolExecutor`
- Thread allocation, task submission, and worker limits
- Chunked streaming of web content to limit RAM consumption during heavy downloads
- Visualizing aggregated transfer rates using `tqdm` progress monitors
- Implementing exponential retry backoff logic manually


In [ ]:
import os
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# List of assets for download
DOWNLOAD_TASKS = [
    ('https://picsum.photos/800/600', 'image_1.jpg'),
    ('https://picsum.photos/1024/768', 'image_2.jpg'),
    ('https://picsum.photos/1920/1080', 'image_3.jpg'),
    ('https://picsum.photos/640/480', 'image_4.jpg'),
    ('https://raw.githubusercontent.com/python/cpython/main/LICENSE', 'python_license.txt'),
    ('https://www.python.org/static/community_logos/python-logo-master-v3-TM.png', 'python_logo.png')
]

def download_asset(url, dest_name, retries=3):
    """Downloads a file with exponential retry mechanism."""
    out_dir = 'downloads'
    os.makedirs(out_dir, exist_ok=True)
    filepath = os.path.join(out_dir, dest_name)
    
    delay = 1.0
    for attempt in range(1, retries + 1):
        try:
            # Stream content in chunks rather than buffering fully in memory
            response = requests.get(url, stream=True, timeout=8)
            response.raise_for_status()
            
            with open(filepath, 'wb') as f:
                for chunk in response.iter_content(chunk_size=4096):
                    if chunk:
                        f.write(chunk)
            return dest_name, True, None
            
        except (requests.RequestException, IOError) as err:
            if attempt == retries:
                return dest_name, False, str(err)
            time.sleep(delay)
            delay *= 2.0  # Double delay every attempt
            
def concurrent_download_manager(tasks, pool_size=4):
    print(f'Starting concurrent downloads using {pool_size} threads...')
    start_time = time.time()
    results = []
    
    with ThreadPoolExecutor(max_workers=pool_size) as executor:
        # Map thread submission to original filenames
        task_map = {executor.submit(download_asset, url, name): name for url, name in tasks}
        
        # Wrap progress display around task futures
        for future in tqdm(as_completed(task_map), total=len(tasks), desc='Overall Progress'):
            filename = task_map[future]
            try:
                fname, success, err = future.result()
                results.append((fname, success, err))
            except Exception as e:
                results.append((filename, False, str(e)))
                
    elapsed = time.time() - start_time
    print(f'\n🏁 Finished all transfers in {elapsed:.2f} seconds.')
    
    # Print final execution table
    print('+' + '-'*50 + '+')
    print(f'| {"Asset":<22} | {"Status":<23} |')
    print('+' + '-'*50 + '+')
    for name, success, error in results:
        status = '✅ Success' if success else f'❌ Failed: {error[:12]}'
        print(f'| {name:<22} | {status:<23} |')
    print('+' + '-'*50 + '+')


In [ ]:
# Execute concurrent downloads with 4 workers
concurrent_download_manager(DOWNLOAD_TASKS, pool_size=4)
